In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from datetime import datetime

In [0]:
silver_df = spark.table("raj.silver.stock_prices")

# silver_df.printSchema()
display(silver_df.limit(10))

In [0]:
# silver_df.withColumn("date_key",date_format(col("trade_date"),"yyyyMMdd").cast("int"))\
dim_date = silver_df.select("trade_date").distinct() 
display(dim_date)

In [0]:
dim_date = dim_date.withColumn("date_key",date_format(col("trade_date"),"yyyyMMdd").cast("int"))\
    .withColumn("full_date",col("trade_date")).withColumn("year",year(col("trade_date")))\
    .withColumn("quarter",quarter(col("trade_date"))).withColumn("month",month(col("trade_date")))\
    .withColumn("month_name",date_format(col("trade_date"),"MMMM")).withColumn("day_of_week",dayofweek(col("trade_date")))\
    .withColumn("day_name",date_format(col("trade_date"),"EEEE")).withColumn("day",dayofmonth(col("trade_date")))\
    .withColumn("week_of_year",weekofyear(col("trade_date")))
display(dim_date)

In [0]:
dim_date = dim_date.withColumn("quarter_name",concat(lit('Q'),col("quarter").cast("string")))\
    .withColumn("is_month_start",when(col("day")==1,True).otherwise(False))\
    .withColumn("is_month_end",when(col("day")==dayofmonth(last_day(col("trade_date"))),True).otherwise(False))\
    .withColumn("is_quarter_start",when(
        (col("month").isin(1,4,7,9)) &
        (col("day")==1),True
    ).otherwise(False))\
    .withColumn("is_quarter_end",when(
        (col("month").isin(3,6,9,12)) &
        (col("day")==dayofmonth(last_day(col("trade_date")))),True
    ).otherwise(False)).drop("trade_date")

# display(dim_date.filter(col("is_month_start")==True).select("trade_date"))
display(dim_date.orderBy(col("date_key")))

In [0]:
dim_date.write.format("delta").mode("overwrite").option("overwriteSchema",True).saveAsTable("raj.gold.dim_date")

In [0]:
verify_table = spark.table("raj.gold.dim_date")

display(verify_table.count())

display(verify_table.filter((col("is_quarter_end")==True) | (col("is_quarter_start")==True)))
